## 1. Импорт библиотек и настройка

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import warnings
warnings.filterwarnings("ignore")

## 2. Загрузка данных и их общее исследование

In [ ]:
df_transaction = pd.read_parquet('/kaggle/input/alfa-challenge/df_transaction.pa')
df_train = pd.read_parquet('/kaggle/input/alfa-challenge/train.pa')

In [ ]:
df_transaction.info()

#### Очень важный момент, что mcc_code и merchant_name относятся к object!

In [ ]:
df_transaction.describe(include='all')

In [ ]:
# пропущенные значения

df_transaction.isnull().sum()

In [ ]:
# Уникальные значения

df_transaction.nunique()

#### Видим, что уникальных mcc_code относительно мало, можно в будущем сделать что-нибудь по типу One Hot Encoding?

## 3. Анализ изменений суммы транзакций (amount) во времени для каждого target

In [ ]:
df_merged = pd.merge(df_transaction, df_train, on='client_num', how='inner')

unique_targets = df_merged['target'].unique()
clients_per_target = {}

for target in unique_targets:
    clients_per_target[target] = df_merged[df_merged['target'] == target]['client_num'].iloc[0]

plt.figure(figsize=(15, 20))

for idx, target_client in enumerate(sorted(clients_per_target.items()), 1):
    target, client_num = target_client
    
    client_data = df_merged[df_merged['client_num'] == client_num]
    client_data = client_data.sort_values(by='date_time')
    
    plt.subplot(len(clients_per_target), 1, idx)
    plt.plot(client_data['date_time'], client_data['amount'], marker='o', label=f'Target: {target}')
    plt.title(f'Изменение amount во времени (Target: {target}, Client: {client_num})')
    plt.xlabel('Дата и время')
    plt.ylabel('Сумма транзакции')
    plt.legend(loc=(1.1, 0.5))
    plt.grid(True)

plt.tight_layout()
plt.show()

## 4. Анализ дубликатов

In [ ]:
# Посмотрим на дубликаты в датасете исходном

df_transaction.duplicated().sum()

In [ ]:
# Конечно дубликаты могут быть из-за того, что один человек может производить финансовые операции в одной и той же категории. 
# Посмотрим может быть зависимость между количеством дубликатов и target?

duplicates = df_merged[df_merged.duplicated()]
duplicates_per_target = duplicates.groupby('target').size().reset_index(name='duplicates_count')

plt.figure(figsize=(10, 6))
plt.bar(duplicates_per_target['target'], duplicates_per_target['duplicates_count'], color='skyblue')
plt.xlabel('Target')
plt.ylabel('Количество дубликатов')
plt.grid(axis='y')
plt.show()

## 5. Анализ распределения классов в целевой переменной

In [ ]:
# Видим что к target 6 количество дубликатов падает. Может это из-за несбалансированности классов? Проверим

target_counts = df_train['target'].value_counts().sort_index()

plt.figure(figsize=(10, 6))
sns.barplot(x=target_counts.index, y=target_counts.values, palette='viridis')
plt.title('Распределение классов в целевой переменной', fontsize=14)
plt.xlabel('Target', fontsize=12)
plt.ylabel('Количество', fontsize=12)
plt.show()

target_percentages = (target_counts / target_counts.sum()) * 100
print("\nПроцентное распределение классов:")
print(target_percentages)

#### Видим, что распределение target и распределение дубликатов между ними совпадает => с большей вероятностью дубликаты и target не связаны

## 6. Анализ транзакций с нулевыми суммами и редких MCC кодов

### Можно было бы посмотреть на выбросы, но на что именно смотреть? 🤔

#### Транзакции с большими суммами можно ли считать выбросами? На мой взгляд это лучше посмотреть на обучении моделей. Если при удалении выбросов (операций с высокими суммами точность увеличится, то, возможно, реально стоит удалить такие данные). Однако обратимся к другой стороне вопроса.
#### Операции с нулевым балансом? Есть ли такие?

In [ ]:
df_transaction[df_transaction['amount'] <= 0]

#### Видим что таких данных 490 строк (что не много в рамках всех данных и, скорее всего, не сильно повлияет на исходные предсказания, так как опять же данных исходных много). Однако что будет, если их удалить? Уменьшиться ли ошибка?

#### Посмотрим на mcc_code, которых очень мало. Много ли таких?

In [ ]:
mcc_counts = df_transaction['mcc_code'].value_counts()

# Отберем те значения, которые встречаются реже, например, менее 10 раз
rare_mcc_codes = mcc_counts[mcc_counts < 10]

plt.figure(figsize=(10, 6))
sns.barplot(x=rare_mcc_codes.index.astype(str), y=rare_mcc_codes.values)
plt.title('Редкие значения MCC-кодов (частота < 10)', fontsize=14)
plt.xlabel('MCC код', fontsize=12)
plt.ylabel('Частота', fontsize=12)
plt.xticks(rotation=90)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

#### Интересно, влияет ли как-то малоиспользуемые mcc коды на target? 🤔💭

#### Посмотрим на среднее количество произведенных финансовых операций по target

## 7. Анализ среднего количества транзакций по target

In [ ]:
transactions_per_target = df_merged.groupby('target')['client_num'].count()
unique_clients_per_target = df_merged.groupby('target')['client_num'].nunique()
avg_transactions_per_target = transactions_per_target / unique_clients_per_target

# Нормализация средних значений для максимального значения в 100%
avg_transactions_percentage = (avg_transactions_per_target / avg_transactions_per_target.sum()) * 100

plt.figure(figsize=(10, 6))
sns.barplot(x=avg_transactions_percentage.index, y=avg_transactions_percentage.values, palette='viridis')

plt.title('Среднее количество транзакций на клиента для каждого target в процентном соотношении', fontsize=14)
plt.xlabel('Target', fontsize=12)
plt.ylabel('Среднее количество транзакций (%)', fontsize=12)
plt.xticks(rotation=90)
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

#### Видим, что у target 0 количество операций в среднем меньше. Почему же у target 6 среднее количество операций падает?

#### Я не нашел точной информации. Однако насколько я понимаю, target 0 - баланс на карте на след месяц стремится к 0.
#### target 6, возможно, падает потому что пользователи делают большие финансовые операции, но редко? Однако выше мы не видели существенных больших операций для target 5, 6, но мы смотреть только для одного пользователя. 
#### Попробуем посмотреть на несколько пользователей

In [ ]:
df_merged = pd.merge(df_transaction, df_train, on='client_num', how='inner')

unique_targets = sorted(df_merged['target'].unique())

num_clients = 5

clients_per_target = {}

for target in unique_targets:
    clients = df_merged[df_merged['target'] == target]['client_num'].unique()

    # Если клиентов больше или равно 5, случайным образом выбираем 5
    if len(clients) >= num_clients:
        selected_clients = np.random.choice(clients, size=num_clients)
    else:
        selected_clients = clients
    clients_per_target[target] = selected_clients

num_targets = len(unique_targets)
fig, axes = plt.subplots(num_targets, num_clients, figsize=(5 * num_clients, 4 * num_targets))

for row_idx, target in enumerate(unique_targets):
    client_nums = clients_per_target[target]

    for col_idx in range(num_clients):
        ax = axes[row_idx, col_idx] if num_targets > 1 else axes[col_idx]
        client_num = client_nums[col_idx]
        client_data = df_merged[df_merged['client_num'] == client_num].sort_values(by='date_time')
        ax.plot(client_data['date_time'], client_data['amount'], marker='o', linestyle='-')
        ax.set_title(f'Client: {client_num}', fontsize=10)
        ax.set_xlabel('Дата и время', fontsize=8)
        ax.set_ylabel('Сумма транзакции', fontsize=8)
        ax.tick_params(axis='x', rotation=90)
        ax.grid(True, linestyle='--', alpha=0.5)

    axes[row_idx, 0].set_ylabel(f'Target: {target}', rotation=90, size='large', labelpad=20)

plt.tight_layout()
plt.show()

#### Как мы можем увидеть, сильную разницу на отдельных примерах между последними таргетами заметить трудно. Так как и в 4, и в 5, и в 6 есть люди которые совершают несколько крупных операций. При этом есть примеры, когда человек target 6 не имеет операций с крупными суммами и при этом не слишком часто в принципе совершает операции. Данный вопрос стоит оставить для более подробного рассмотрения (+ с помощью ML моделей, так как глаз человека может не заметить разницу и не может просмотреть все варианты) 

## 8. Анализ временных тенденций транзакций

In [ ]:
df_merged_copy = df_merged.copy()

df_merged_copy['date_time'] = pd.to_datetime(df_merged['date_time'])

df_merged_copy['date'] = df_merged['date_time'].dt.date

daily_trends_copy = df_merged_copy.groupby(['date', 'target'])['amount'].sum().reset_index()

plt.figure(figsize=(15, 10))
for target in sorted(daily_trends_copy['target'].unique()):
    target_data = daily_trends_copy[daily_trends_copy['target'] == target]
    plt.plot(target_data['date'], target_data['amount'], label=f'Target: {target}')

plt.title('Изменение суммы транзакций по времени для различных target (df_merged_copy)')
plt.xlabel('Дата')
plt.ylabel('Сумма транзакций')
plt.legend(title='Target')
plt.xticks(rotation=90)
plt.grid(True)
plt.tight_layout()
plt.show()

#### Теперь мы можем отчетливо увидеть, что люди, отнесенные к target 6 совершают меньше транзакций, однако более с большими суммами

## 9. Анализ значимости MCC кодов

### P.S. На мой взгляд лучше было бы использовать Permutations importance или Shap Values (более точные), однако для общей информации будет достаточно кода ниже

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

df_mcc_target = df_merged[['mcc_code', 'target']].copy()
df_encoded = pd.get_dummies(df_mcc_target['mcc_code'], prefix='mcc')
df_encoded['target'] = df_mcc_target['target']
X = df_encoded.drop('target', axis=1)
y = df_encoded['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)

feature_importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({'mcc_code': X.columns, 'importance': feature_importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False).reset_index(drop=True)

top_5_mcc = feature_importance_df.head(5)
bottom_5_mcc = feature_importance_df.tail(5)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='mcc_code', data=top_5_mcc, palette='viridis')
plt.title('Топ-5 самых влиятельных mcc_code', fontsize=16)
plt.xlabel('Важность признака', fontsize=14)
plt.ylabel('mcc_code', fontsize=14)
for index, value in enumerate(top_5_mcc['importance']):
    plt.text(value, index, f"{value:.4f}", va='center')
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='mcc_code', data=bottom_5_mcc, palette='magma')
plt.title('Топ-5 наименее влиятельных mcc_code', fontsize=16)
plt.xlabel('Важность признака', fontsize=14)
plt.ylabel('mcc_code', fontsize=14)
for index, value in enumerate(bottom_5_mcc['importance']):
    plt.text(value, index, f"{value:.6f}", va='center')
plt.show()

## 10. Анализ регулярности клиентов по target 

#### Есть ли отличие между регулярностью клиентов в разных target?

In [ ]:
df_merged['date_time'] = pd.to_datetime(df_merged['date_time'])

df_merged = df_merged.sort_values(['client_num', 'date_time'])

# Функция для вычисления регулярности
def compute_regularity(group):
    group = group.sort_values('date_time')
    group['diff'] = group['date_time'].diff().dt.days
    total_days = (group['date_time'].max() - group['date_time'].min()).days
    num_transactions = group.shape[0]
    avg_interval = group['diff'].mean()
    std_interval = group['diff'].std()
    return pd.Series({
        'num_transactions': num_transactions,
        'total_days': total_days,
        'avg_interval_days': avg_interval,
        'std_interval_days': std_interval
    })

regularity_df = df_merged.groupby('client_num').apply(compute_regularity).reset_index()

regularity_df = regularity_df.merge(df_train[['client_num', 'target']], on='client_num', how='left')

# Удаление клиентов с менее чем 2 транзакциями
regularity_df = regularity_df[regularity_df['num_transactions'] > 1]

plt.figure(figsize=(12, 8))
sns.boxplot(x='target', y='avg_interval_days', data=regularity_df, palette='Set3')
plt.title('Средний интервал между транзакциями по target', fontsize=16)
plt.xlabel('Target', fontsize=14)
plt.ylabel('Средний интервал (дни)', fontsize=14)
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(12, 8))
sns.boxplot(x='target', y='std_interval_days', data=regularity_df, palette='Set2')
plt.title('Стандартное отклонение интервала между транзакциями по target', fontsize=16)
plt.xlabel('Target', fontsize=14)
plt.ylabel('Стандартное отклонение (дни)', fontsize=14)
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(12, 8))
sns.boxplot(x='target', y='num_transactions', data=regularity_df, palette='Set1')
plt.title('Количество транзакций по target', fontsize=16)
plt.xlabel('Target', fontsize=14)
plt.ylabel('Количество транзакций', fontsize=14)
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(12, 8))
sns.boxplot(x='target', y='total_days', data=regularity_df, palette='Set3')
plt.title('Общий период активности клиента по target', fontsize=16)
plt.xlabel('Target', fontsize=14)
plt.ylabel('Период активности (дни)', fontsize=14)
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(14, 10))
sns.kdeplot(data=regularity_df, x='avg_interval_days', hue='target', fill=True)
plt.title('Распределение среднего интервала между транзакциями по target', fontsize=16)
plt.xlabel('Средний интервал (дни)', fontsize=14)
plt.ylabel('Плотность', fontsize=14)
plt.show()

plt.figure(figsize=(14, 10))
sns.kdeplot(data=regularity_df, x='std_interval_days', hue='target', fill=True)
plt.title('Распределение стандартного отклонения интервала между транзакциями по target', fontsize=16)
plt.xlabel('Стандартное отклонение (дни)', fontsize=14)
plt.ylabel('Плотность', fontsize=14)
plt.show()